In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 7.4 Text Mining in Higher Ed – Topic Modeling
- Unsupervised: no pre-labeled data needed
- NMF (on TF-IDF) vs LDA (on counts)
- Assigning a Dominant Topic per student

## Setup

In [ ]:
import pandas as pd

ML_Survey_Data = pd.read_csv('../data/ML_Survey_Data.csv')
display(ML_Survey_Data)

## TF-IDF Vectors (Recap)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2), min_df=2)
tfidf_matrix = tfidf_vec.fit_transform(ML_Survey_Data['Free_Response_Text'])
feature_names_tfidf = tfidf_vec.get_feature_names_out()
print("TF-IDF matrix:", tfidf_matrix.shape)

## NMF Topic Modeling
NMF factorizes the document-term matrix into a Document-Topic matrix (W) and a Topic-Term matrix (H). Crisp, distinct topics — good for short survey comments.

In [ ]:
from sklearn.decomposition import NMF

N_TOPICS = 4
nmf_model = NMF(n_components=N_TOPICS, random_state=42, max_iter=500)
W_nmf = nmf_model.fit_transform(tfidf_matrix)
H_nmf = nmf_model.components_

def print_top_words(H, feature_names, n_words=8, label='Topic'):
    for i, row in enumerate(H):
        top_idx = row.argsort()[:-n_words-1:-1]
        print(f"  {label} {i}: {', '.join(feature_names[j] for j in top_idx)}")

print("NMF Topics:")
print_top_words(H_nmf, feature_names_tfidf)

## LDA Topic Modeling
LDA is probabilistic — treats each response as a mixture of topics. Runs on raw counts, not TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

count_vec = CountVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2), min_df=2)
count_matrix = count_vec.fit_transform(ML_Survey_Data['Free_Response_Text'])
feature_names_count = count_vec.get_feature_names_out()

lda_model = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=20)
W_lda = lda_model.fit_transform(count_matrix)

print("LDA Topics:")
print_top_words(lda_model.components_, feature_names_count)

## Assigning Dominant Topics
Use `np.argmax` on the document-topic matrix to turn topic weights into a single categorical feature per student.

In [ ]:
import numpy as np

df_Topic_Modeling = ML_Survey_Data[['SID', 'Free_Response_Text']].copy()
df_Topic_Modeling['Dominant_Topic_NMF'] = np.argmax(W_nmf, axis=1)
df_Topic_Modeling['Dominant_Topic_LDA'] = np.argmax(W_lda, axis=1)

print(df_Topic_Modeling['Dominant_Topic_NMF'].value_counts().sort_index())
df_Topic_Modeling

## Summary
- NMF → crisp topics from TF-IDF. LDA → probabilistic mixtures from raw counts.
- Dominant topic per student = the categorical feature ready for downstream modeling or reporting.

**Next:** 7.5 estimates the emotional tone of these same comments.